# B-11: monthly-resolution rollout + downscaling back to daily

B-09/B-10 (D-53/D-54) found recursive daily rollout's R2 stays modest because models miss CH4's
large spikes, even though MASE mostly beats persistence. The M5 competition's own hierarchy
findings show coarser aggregates score better (a single missed spike-day matters much less to a
monthly mean than to that one day's value) -- this notebook tests whether that holds here, and
whether the improvement survives being downscaled back to a daily series comparable to B-09/B-10.

**Design**: `src/features/build_forecasting_matrix_monthly.py` resamples `forecast_daily_v2.csv`
up to monthly (sum for precip, mean for most `fx_` columns, min/max for TA, fraction-of-month for
binary flags, month-end value for the `days_since_grazing` counter, DOY sin/cos recomputed fresh
from month-of-year, soil lags/rolls re-lagged at monthly windows, new `ar_ch4_mlag{1,2,3}`) into
`forecast_monthly_v2.csv`. **Model roster: SARIMAX + RF/XGB/LightGBM only** -- DLinear/LSTM scoped
out (only ~90 monthly rows per tower, too thin for a from-scratch DL window regime).

**Anchor alignment**: the daily anchor (2021-12-16) has no clean monthly equivalent, so the
monthly anchor is the last *fully complete* month before it (2021-11-01) -- using November 2021
as the last real training month avoids leaking December 2021's post-anchor days into "training".
13 months are then forecast (Dec 2021 -- Dec 2022), covering the same span as B-09/B-10's 365-day
daily window.

**Downscaling (the core new mechanism) -- hybrid-calibration, not independent of the daily
models' own shape errors**: `recursive_rollout.downscale_monthly_to_daily(monthly_pred,
daily_template)` reuses the **corresponding B-09 daily chain for the same anchor/tower/model** as
the within-month shape template, then recenters it so its own monthly mean matches this
notebook's independently-derived monthly prediction:
`daily_synth[d] = daily_template[d] - mean(daily_template over month) + monthly_pred[month]`.
**This means any within-month spike-miss error is inherited unchanged from the daily template --
only the month-to-month bias gets corrected.** State this caveat plainly, not as a footnote.

In [1]:
from pathlib import Path
import sys, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
sys.path.insert(0, "../../src")

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import r2_score, mean_absolute_error

import models.recursive_rollout as rr
from evaluation.metrics import mase as mase_fn

HOURLY = Path("../../data/Hourly"); RESULTS = Path("../../results")
TOWER = 4
N_MONTHS = 13
DAILY_N_DAYS = 365
AR_COLS_M = ["ar_ch4_mlag1", "ar_ch4_mlag2", "ar_ch4_mlag3"]
EXOG_B = ["fx_lsu_dens", "fx_WS_mean", "fx_VPD_mean", "fx_USTAR_mean", "fx_PPFD_mean",
          "fx_DOY_sin", "fx_DOY_cos", "fx_is_growing"]
DAILY_ANCHOR = pd.Timestamp("2021-12-16")
MONTHLY_ANCHOR = pd.Timestamp("2021-11-01")
print(f"Monthly anchor: {MONTHLY_ANCHOR.date()}  daily anchor: {DAILY_ANCHOR.date()}")

Monthly anchor: 2021-11-01  daily anchor: 2021-12-16


## 1  Load monthly data (built by `build_forecasting_matrix_monthly.py`)

In [2]:
dm = pd.read_csv(HOURLY/"forecast_monthly_v2.csv", low_memory=False)
dm["Datetime"] = pd.to_datetime(dm["Datetime"], format="mixed")
FX_M = [c for c in dm.columns if c.startswith("fx")]
M = {t: dm[dm.tower == t].set_index("Datetime").sort_index() for t in [2, 4, 9]}
print(f"Tower 4: {len(M[TOWER])} months, {int(M[TOWER]['y_observed'].notna().sum())} with real observed data")

Tower 4: 97 months, 72 with real observed data


## 2  Tree models (RandomForest, XGBoost, LightGBM), monthly resolution

Same hyperparameters and pooled-training convention as B-09/B-10 (no new HPO, D-41) -- only the
feature set (`ar_ch4_mlag*` instead of `ar_ch4_dlag*`) and step size (`pd.DateOffset(months=1)`
instead of `pd.Timedelta(days=1)`) differ.

In [3]:
def fit_tree(algo, tr, feat_cols):
    imp = SimpleImputer(strategy="mean"); Xi = imp.fit_transform(tr[feat_cols].values)
    if algo == "RF":
        m = RandomForestRegressor(n_estimators=500, n_jobs=-1, random_state=42,
                                   min_samples_leaf=10, max_features=0.5)
    elif algo == "XGB":
        m = XGBRegressor(subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42,
                          max_depth=2, learning_rate=0.02, n_estimators=400, min_child_weight=10)
    elif algo == "LightGBM":
        m = LGBMRegressor(subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42,
                           num_leaves=7, min_child_samples=10, learning_rate=0.02, n_estimators=400,
                           verbosity=-1)
    else:
        raise ValueError(algo)
    m.fit(Xi, tr["target"].values); return m, imp

feat_cols = AR_COLS_M + FX_M + ["is_t2", "is_t4", "is_t9"]
pool = []
for t in [2, 4, 9]:
    df = M[t].copy(); df["target"] = df["y_gapfilled"]
    pool.append(df[df.index <= MONTHLY_ANCHOR])
tr = pd.concat(pool); tr = tr[tr["target"].notna()]
print(f"Pooled monthly training rows (<= anchor): {len(tr)}")

dm4 = M[TOWER]
history_init = dm4.loc[:MONTHLY_ANCHOR, "y_gapfilled"].copy()
target_months = pd.date_range(MONTHLY_ANCHOR + pd.DateOffset(months=1), periods=N_MONTHS, freq="MS")
fx_frame = dm4.loc[target_months, FX_M].copy()
fx_frame["is_t2"], fx_frame["is_t4"], fx_frame["is_t9"] = 0.0, 1.0, 0.0

tree_chains = {}
for algo in ["RF", "XGB", "LightGBM"]:
    t0 = time.time()
    model, imp = fit_tree(algo, tr, feat_cols)
    tree_chains[algo] = rr.monthly_rollout(model, imp, feat_cols, fx_frame, history_init,
                                            MONTHLY_ANCHOR, n_months=N_MONTHS)
    print(f"  {algo} monthly rollout done ({time.time()-t0:.1f}s)", flush=True)

Pooled monthly training rows (<= anchor): 177


  RF monthly rollout done (0.8s)


  XGB monthly rollout done (0.2s)


  LightGBM monthly rollout done (0.0s)


## 3  SARIMAX, monthly resolution

In [4]:
y = dm4["y_gapfilled"].astype(float)
X = dm4[EXOG_B].astype(float).ffill().bfill()
y_tr, X_tr = y.loc[:MONTHLY_ANCHOR], X.loc[:MONTHLY_ANCHOR]

best = None
for p in [1, 2]:
    for q in [0, 1]:
        try:
            m = SARIMAX(y_tr, exog=X_tr, order=(p, 1, q), enforce_stationarity=False, enforce_invertibility=False)
            res = m.fit(disp=False, maxiter=50)
            if best is None or res.aic < best[0]: best = (res.aic, (p, 1, q), res)
        except Exception:
            continue
sarimax_order, sarimax_res = best[1], best[2]
future_X = X.loc[target_months]
fc = sarimax_res.get_forecast(steps=N_MONTHS, exog=future_X)
sarimax_chain = pd.Series(fc.predicted_mean.values, index=target_months)
all_chains = {**tree_chains, "SARIMAX": sarimax_chain}
print(f"SARIMAX order={sarimax_order}")

C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency 

C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency 

SARIMAX order=(1, 1, 1)


## 4  Monthly-native sanity check

Evaluated directly against real monthly `y_observed` (mean of daily `y_observed` within each
month) -- the M5-lesson prediction is that this should score much better than B-09's daily-native
numbers, simply because a monthly mean is far less sensitive to a single missed spike-day.

In [5]:
anchor_val = dm4.loc[MONTHLY_ANCHOR, "y_gapfilled"]
persist_m = np.full(N_MONTHS, float(anchor_val))
y_true_m = dm4.loc[target_months, "y_observed"].values
ok = np.isfinite(y_true_m)

rows = []
for name, chain in all_chains.items():
    yp = chain.reindex(target_months).values
    r2 = r2_score(y_true_m[ok], yp[ok]) if ok.sum() > 2 and np.var(y_true_m[ok]) > 0 else np.nan
    mae_v = mean_absolute_error(y_true_m[ok], yp[ok]) if ok.sum() > 2 else np.nan
    mase_v = mase_fn(y_true_m[ok], yp[ok], persist_m[ok]) if ok.sum() > 2 else np.nan
    rows.append(dict(model=name, n=int(ok.sum()), R2=round(r2, 3) if np.isfinite(r2) else np.nan,
                      MAE=round(float(mae_v), 3), MASE=round(float(mase_v), 4) if np.isfinite(mase_v) else np.nan))
R_monthly = pd.DataFrame(rows)
R_monthly.to_csv(RESULTS/"b11_monthly_summary.csv", index=False)
print(R_monthly.to_string(index=False))

   model  n    R2    MAE   MASE
      RF 13 0.553 21.705 0.7427
     XGB 13 0.602 21.126 0.7229
LightGBM 13 0.619 20.553 0.7033
 SARIMAX 13 0.531 22.539 0.7712


## 5  Downscale to daily, evaluate on the same `bin_metrics` framework as B-09/B-10

Reuses B-09's own daily chain (`results/b09_chains.csv`) for this same anchor/tower/model as the
within-month shape template. Verifies the recentering is exact by construction (monthly mean of
the downscaled daily series must equal the input monthly prediction, for every month) before
trusting the result.

In [6]:
b09_chains = pd.read_csv(RESULTS/"b09_chains.csv", index_col=0, parse_dates=True)
daily_target_dates = pd.date_range(DAILY_ANCHOR + pd.Timedelta(days=1), periods=DAILY_N_DAYS, freq="D")
y_true_daily = b09_chains["y_true"].reindex(daily_target_dates).values

dv = pd.read_csv(HOURLY/"forecast_daily_v2.csv", low_memory=False)
dv["Datetime"] = pd.to_datetime(dv["Datetime"], format="mixed")
dv4 = dv[dv.tower == TOWER].set_index("Datetime").sort_index()
anchor_val_daily = dv4.loc[DAILY_ANCHOR, "y_gapfilled"]
persist_daily = rr.chain_persistence(anchor_val_daily, DAILY_N_DAYS)

rows = []
for name, chain in all_chains.items():
    template = b09_chains[name].reindex(daily_target_dates)
    daily_synth = rr.downscale_monthly_to_daily(chain, template)

    # exactness check: recentered daily series' monthly mean must equal the input monthly pred
    check = daily_synth.groupby(daily_synth.index.to_period("M")).mean()
    chain_by_period = chain.copy(); chain_by_period.index = chain_by_period.index.to_period("M")
    max_diff = (check - chain_by_period.reindex(check.index)).abs().max()
    print(f"  {name}: downscale recenter exactness check, max abs diff = {max_diff:.10f} (expect 0)")

    yp = daily_synth.reindex(daily_target_dates).values
    bm = rr.bin_metrics(y_true_daily, yp, daily_target_dates, DAILY_ANCHOR, y_persist=persist_daily)
    bm["model"] = name
    rows.append(bm)
R_downscaled = pd.concat(rows, ignore_index=True)
R_downscaled.to_csv(RESULTS/"b11_downscaled_summary.csv", index=False)

pd.set_option("display.width", 200)
print("\n=== downscaled-to-daily R2 by model x bin ===")
print(R_downscaled.pivot_table(index="model", columns="bin", values="R2").round(3).to_string())
print("\n=== B-09's original daily-native R2 by model x bin, for comparison ===")
rows_orig = []
for name in ["RF", "XGB", "LightGBM", "SARIMAX"]:
    yp = b09_chains[name].reindex(daily_target_dates).values
    bm = rr.bin_metrics(y_true_daily, yp, daily_target_dates, DAILY_ANCHOR, y_persist=persist_daily)
    bm["model"] = name
    rows_orig.append(bm)
R_orig = pd.concat(rows_orig, ignore_index=True)
print(R_orig.pivot_table(index="model", columns="bin", values="R2").round(3).to_string())

  RF: downscale recenter exactness check, max abs diff = 0.0000000000 (expect 0)
  XGB: downscale recenter exactness check, max abs diff = 0.0000000000 (expect 0)
  LightGBM: downscale recenter exactness check, max abs diff = 0.0000000000 (expect 0)
  SARIMAX: downscale recenter exactness check, max abs diff = 0.0000000000 (expect 0)

=== downscaled-to-daily R2 by model x bin ===
bin         1-7  181-270  271-365  31-90   8-30  91-180
model                                                  
LightGBM -0.753    0.368    0.046 -0.307 -0.015   0.092
RF       -4.047    0.253    0.019 -0.104 -0.251   0.095
SARIMAX  -3.728    0.167   -0.102 -0.308  0.043   0.104
XGB      -0.581    0.364    0.044 -0.199 -0.007   0.062

=== B-09's original daily-native R2 by model x bin, for comparison ===
bin         1-7  181-270  271-365  31-90   8-30  91-180
model                                                  
LightGBM -0.606    0.335   -0.582 -0.056 -0.214   0.193
RF       -3.127    0.222   -0.594 -0.182 

## 6  Multi-anchor (2018-2022) extension -- the actual verdict

Per B-09/B-10's own lesson, single-anchor results above are a smoke test only -- run as a script
extension (`b11_multi_anchor.py`, not re-executed here, same precedent as B-09/B-10), reusing each
anchor year's own B-09 daily chain (`results/b09_chains_anchor{yr}.csv`) as that year's downscaling
template. Results: `results/b11_monthly_multi_anchor.csv`, `results/b11_downscaled_multi_anchor.csv`.
Full interpretation in `b11_results.md`. Headline (mean R2/MASE across 5 anchors):

- **Monthly-native evaluation is a real, substantial improvement** over B-09's daily-native
  numbers: XGB mean R2=0.147 (vs 0.003 daily), LightGBM 0.156 (vs -0.014), confirming the M5-lesson
  prediction that coarser aggregation dampens spike-miss error.
- **This improvement does NOT survive downscaling back to daily.** Downscaled-to-daily mean R2:
  XGB -0.000, LightGBM -0.016, RF -0.064, SARIMAX -0.244 -- essentially unchanged from (or, for
  SARIMAX, worse than) B-09's own daily-native numbers (0.003/-0.014/-0.067/-0.039). By bin, the
  late-window bins (181-270, 271-365) improve modestly (SARIMAX 271-365 flips -0.271->-0.037,
  RF/XGB/LightGBM all reach ~0.05-0.06 there) but the short/mid bins (1-7, 8-30) get *worse* than
  B-09's originals when averaged across 5 anchors.
- **Why**: the downscaling method reuses the daily template's own within-month *shape* unchanged,
  correcting only the month-to-month *bias*. Since B-09's daily models already capture the coarse
  seasonal trend reasonably (they train on AR + seasonal `fx_` features), the correction adds
  little where the daily model was already roughly unbiased, and can subtract signal where the
  monthly model's own seasonal fit disagrees with the daily model's.

## 7  Append to benchmarks.csv (B11, single-anchor smoke-test rows)

In [7]:
bench = RESULTS/"benchmarks.csv"; today = pd.Timestamp.today().date().isoformat()
ex = pd.read_csv(bench); ex = ex[ex["replication"] != "B11"]
rows = []
for _, r in R_downscaled.iterrows():
    lo, hi = r["bin"].split("-")
    rows.append({"replication": "B11", "model": r["model"], "tower": f"Tower {TOWER}",
        "feature_set": "monthly rollout downscaled to daily (hybrid-calibration, B09 daily chain as shape template); single-anchor smoke test 2021-12-16",
        "track": "B", "horizon": int(hi), "split": f"b11_downscaled_leadtime_bin_{r['bin']}",
        "R2": r["R2"], "MAE": r["MAE"], "n_test": int(r["n"]),
        "MASE": r["MASE"], "date": today,
        "notes": f"B11 monthly rollout + downscale-to-daily; SARIMAX order={sarimax_order}; lead-time bin days {r['bin']}; D-55"})
new = pd.DataFrame(rows); comb = pd.concat([ex, new], ignore_index=True); comb.to_csv(bench, index=False)
print(f"Wrote {len(new)} B11 rows. Total {len(comb)}.")

Wrote 24 B11 rows. Total 3911.
